In [3]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="aws-1-us-west-1.pooler.supabase.com",
    port=5432,
    database="postgres",
    user="postgres.xemmevpltwdwcevdnjwg",
    password="********"
)

print("Conexión exitosa")

Conexión exitosa


1. Ranking general de jugadores por puntos acumulados

In [4]:
query1 = """
SELECT
    j.nombre,
    j.pais,
    j.elo,
    r.victorias,
    r.derrotas,
    r.empates,
    r.puntos,
    RANK() OVER (ORDER BY r.puntos DESC) AS posicion_ranking
FROM rankings r
JOIN jugadores j ON r.id_jugador = j.id_jugador
ORDER BY posicion_ranking;
"""

df1 = pd.read_sql(query1, conn)
print("Ranking Global")
df1

Ranking Global


/tmp/ipykernel_1851/1695316844.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query1, conn)


,nombre,pais,elo,victorias,derrotas,empates,puntos,posicion_ranking
0,Magnus Carlsen,Noruega,2830,4,0,0,8,1
1,Wei Yi,China,2748,3,0,1,7,2
2,Shakhriyar Mamedyarov,Azerbaiyan,2732,3,0,1,7,2
3,Alireza Firouzja,Francia,2755,3,1,0,6,4
4,Teimour Radjabov,Azerbaiyan,2710,3,1,0,6,4
5,Gukesh Dommaraju,India,2785,3,1,0,6,4
6,Fabiano Caruana,Estados Unidos,2781,2,1,1,5,7
7,Nodirbek Abdusattorov,Uzbekistan,2765,2,1,1,5,7
8,Hikaru Nakamura,Estados Unidos,2789,2,1,1,5,7
9,Praggnanandhaa,India,2768,1,1,2,4,10


2. Top 10 jugadores con mayor cantidad de victorias

In [6]:
query2 = """
SELECT
    j.nombre,
    j.pais,
    j.elo,
    r.victorias,
    r.derrotas,
    r.empates
FROM rankings r
JOIN jugadores j ON r.id_jugador = j.id_jugador
ORDER BY r.victorias DESC
LIMIT 10;
"""
df2 = pd.read_sql(query2, conn)
df2.columns = ['Nombre', 'País', 'ELO', 'Victorias', 'Derrotas', 'Empates']
print("═" * 50)
print("   TOP 10 JUGADORES CON MAYOR CANTIDAD DE VICTORIAS")
print("═" * 50)
df2

══════════════════════════════════════════════════
   TOP 10 JUGADORES CON MAYOR CANTIDAD DE VICTORIAS
══════════════════════════════════════════════════


/tmp/ipykernel_1851/1442588125.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df2 = pd.read_sql(query2, conn)


,Nombre,País,ELO,Victorias,Derrotas,Empates
0,Magnus Carlsen,Noruega,2830,4,0,0
1,Alireza Firouzja,Francia,2755,3,1,0
2,Teimour Radjabov,Azerbaiyan,2710,3,1,0
3,Gukesh Dommaraju,India,2785,3,1,0
4,Wei Yi,China,2748,3,0,1
5,Shakhriyar Mamedyarov,Azerbaiyan,2732,3,0,1
6,Fabiano Caruana,Estados Unidos,2781,2,1,1
7,Hikaru Nakamura,Estados Unidos,2789,2,1,1
8,Nodirbek Abdusattorov,Uzbekistan,2765,2,1,1
9,Jan-Krzysztof Duda,Polonia,2741,2,2,0


3. Aperturas más usadas y tasa de victoria para piezas blancas

In [7]:
query3 = """
SELECT
    apertura,
    COUNT(*) AS total_partidas,
    SUM(CASE WHEN resultado = 'blancas' THEN 1 ELSE 0 END) AS victorias_blancas,
    ROUND(
        SUM(CASE WHEN resultado = 'blancas' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    ) AS tasa_victoria_blancas
FROM partidas
GROUP BY apertura
ORDER BY total_partidas DESC;
"""
df3 = pd.read_sql(query3, conn)
df3.columns = ['Apertura', 'Total Partidas', 'Victorias Blancas', 'Tasa Victoria Blancas (%)']
print("═" * 50)
print("   APERTURAS MÁS USADAS Y TASA DE VICTORIA")
print("═" * 50)
df3

══════════════════════════════════════════════════
   APERTURAS MÁS USADAS Y TASA DE VICTORIA
══════════════════════════════════════════════════


/tmp/ipykernel_1851/3932689647.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df3 = pd.read_sql(query3, conn)


,Apertura,Total Partidas,Victorias Blancas,Tasa Victoria Blancas (%)
0,Apertura Italiana,64,31,48.44
1,Defensa Francesa,54,22,40.74
2,Apertura Catalana,49,24,48.98
3,Gambito de Dama,46,17,36.96
4,Defensa India de Rey,45,20,44.44
5,Apertura Española,44,14,31.82
6,Apertura Inglesa,43,15,34.88
7,Gambito de Rey,40,13,32.50
8,Apertura Escocesa,40,10,25.00
9,Defensa Caro-Kann,40,14,35.00


4.Torneos con mayor actividad y promedio de movimientos

In [8]:
query4 = """
SELECT
    t.nombre,
    t.ciudad,
    COUNT(p.id_partida) AS total_partidas,
    ROUND(AVG(p.cantidad_movimientos), 2) AS promedio_movimientos
FROM torneos t
JOIN partidas p ON t.id_torneo = p.id_torneo
GROUP BY t.id_torneo, t.nombre, t.ciudad
ORDER BY total_partidas DESC;
"""
df4 = pd.read_sql(query4, conn)
df4.columns = ['Torneo', 'Ciudad', 'Total Partidas', 'Promedio Movimientos']
print("═" * 50)
print("   TORNEOS CON MAYOR ACTIVIDAD Y PROMEDIO DE MOVIMIENTOS")
print("═" * 50)
df4

══════════════════════════════════════════════════
   TORNEOS CON MAYOR ACTIVIDAD Y PROMEDIO DE MOVIMIENTOS
══════════════════════════════════════════════════


/tmp/ipykernel_1851/1644376607.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df4 = pd.read_sql(query4, conn)


,Torneo,Ciudad,Total Partidas,Promedio Movimientos
0,Grand Prix Finals,Dubai,37,51.03
1,Continental Chess League,Buenos Aires,34,59.56
2,International Open,Madrid,34,53.79
3,Blitz Arena,Tokyo,31,47.65
4,World Chess Championship,Londres,31,57.90
5,Torneo de Zúrich,Zúrich,30,58.03
6,Elite Chess Cup,New York,30,55.87
7,Isle of Man Open,Douglas,30,52.67
8,Torneo de Linares,Linares,30,54.43
9,Torneo Candidatos Medellín,Medellín,30,49.10


5. Top 3 jugadores por torneo

In [9]:
query5 = """
WITH ranking_por_torneo AS (
    SELECT
        t.nombre AS torneo,
        j.nombre AS jugador,
        COUNT(*) AS partidas_jugadas,
        SUM(CASE WHEN p.ganador = j.id_jugador THEN 1 ELSE 0 END) AS victorias,
        RANK() OVER (
            PARTITION BY t.id_torneo
            ORDER BY SUM(CASE WHEN p.ganador = j.id_jugador THEN 1 ELSE 0 END) DESC
        ) AS posicion
    FROM partidas p
    JOIN torneos t ON p.id_torneo = t.id_torneo
    JOIN jugadores j ON j.id_jugador = p.jugador_blancas
                     OR j.id_jugador = p.jugador_negras
    GROUP BY t.id_torneo, t.nombre, j.id_jugador, j.nombre
)
SELECT torneo, jugador, partidas_jugadas, victorias, posicion
FROM ranking_por_torneo
WHERE posicion <= 3
ORDER BY torneo, posicion;
"""
df5 = pd.read_sql(query5, conn)
df5.columns = ['Torneo', 'Jugador', 'Partidas Jugadas', 'Victorias', 'Posición']
print("═" * 50)
print("   TOP 3 JUGADORES POR TORNEO")
print("═" * 50)
df5

══════════════════════════════════════════════════
   TOP 3 JUGADORES POR TORNEO
══════════════════════════════════════════════════


/tmp/ipykernel_1851/412196315.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df5 = pd.read_sql(query5, conn)


,Torneo,Jugador,Partidas Jugadas,Victorias,Posición
0,Blitz Arena,Santosh Gujrathi,6,4,1
1,Blitz Arena,Daniil Dubov,5,3,2
2,Blitz Arena,Magnus Carlsen,6,2,3
3,Blitz Arena,Shakhriyar Mamedyarov,4,2,3
4,Blitz Arena,Maxime Vachier-Lagrave,7,2,3
...,...,...,...,...,...
102,Torneo Tal Memorial,Aravindh Chithambaram,4,2,3
103,Torneo Tal Memorial,Daniil Dubov,3,2,3
104,World Chess Championship,Dmitri Sokolov,6,4,1
105,World Chess Championship,Wesley So,7,3,2
